# Widgets

xeus-quickjs implements the protocol of [Jupyter widgets](https://ipywidgets.readthedocs.io), with an API
close to the one of ipywidgets. The widgets are available in `Jupyter.widgets`.

In [ ]:
const {
  IntSlider, FloatSlider, IntRangeSlider, IntProgress, Play,
  Text, Textarea, Checkbox, ToggleButtons, Dropdown, ColorPicker,
  Button, Output, HTML, Label, HBox, VBox, Tab, Accordion, jslink
} = Jupyter.widgets;

## Sliders

A widget is displayed when it is the result of a cell, or with `display`.

In [ ]:
new IntSlider({ value: 7, min: 0, max: 10, description: 'Integer' })

In [ ]:
const temperature = new FloatSlider({ value: 21.5, min: -10, max: 40, step: 0.5, description: 'Temperature' });
display(temperature);

The properties of a widget are synchronized with the frontend:

In [ ]:
temperature.value

In [ ]:
temperature.value = 30;

In [ ]:
new IntRangeSlider({ value: [20, 60], min: 0, max: 100, description: 'Range' })

## Reacting to changes

`observe` calls a function when a property changes.

In [ ]:
const celsius = new FloatSlider({ value: 20, min: -40, max: 100, description: '°C' });
const fahrenheit = new HTML();
const convert = () => {
  fahrenheit.value = `<b>${(celsius.value * 9 / 5 + 32).toFixed(1)} °F</b>`;
};
celsius.observe(convert, 'value');
convert();

new HBox({ children: [celsius, fahrenheit] })

## Buttons and outputs

An `Output` widget captures the outputs of the callbacks.

In [ ]:
const button = new Button({ description: 'Roll the dice', button_style: 'primary', icon: 'dice' });
const output = new Output();
let rolls = [];

button.onClick(output.capture(() => {
  const value = 1 + Math.floor(Math.random() * 6);
  rolls.push(value);
  console.log(`Rolled a ${value}, the average is ${(rolls.reduce((a, b) => a + b) / rolls.length).toFixed(2)}`);
}));

new VBox({ children: [button, output] })

## Text and selection

In [ ]:
const nameInput = new Text({ placeholder: 'Type your name', description: 'Name' });
const language = new Dropdown({ options: ['JavaScript', 'Python', 'Lua', 'C++'], value: 'JavaScript', description: 'Language' });
const greeting = new Label();
const greet = () => {
  greeting.value = nameInput.value ? `${nameInput.value} likes ${language.value}` : '';
};
nameInput.observe(greet, 'value');
language.observe(greet, 'value');

new VBox({ children: [nameInput, language, greeting] })

In [ ]:
new HBox({ children: [
  new Checkbox({ value: true, description: 'Enabled' }),
  new ToggleButtons({ options: ['Small', 'Medium', 'Large'], value: 'Medium' }),
  new ColorPicker({ value: '#f7df1e', description: 'Color', concise: true })
] })

## Linking widgets

`jslink` links two properties in the frontend, without going through the kernel.

In [ ]:
const play = new Play({ value: 0, min: 0, max: 100, step: 1, interval: 50, description: 'Play' });
const linked = new IntSlider({ description: 'Linked' });
jslink([play, 'value'], [linked, 'value']);

new HBox({ children: [play, linked] })

## Progress bars

In [ ]:
const bar = new IntProgress({ value: 0, max: 100, description: 'Loading', bar_style: 'info' });
display(bar);

for (let value = 0; value <= 100; value += 5) {
  bar.value = value;
  await new Promise(resolve => setTimeout(resolve, 60));
}
bar.bar_style = 'success';
bar.description = 'Done';

## Containers

In [ ]:
new Tab({
  children: [
    new Textarea({ value: 'Some notes', rows: 4 }),
    new VBox({ children: [new Checkbox({ description: 'Option A' }), new Checkbox({ description: 'Option B' })] })
  ],
  titles: ['Notes', 'Options']
})

In [ ]:
new Accordion({
  children: [new Label({ value: 'The first section' }), new Label({ value: 'The second section' })],
  titles: ['First', 'Second']
})